# ML-only comment features — **standalone notebook**

**No Git clone.** No subprocess to repo scripts. This file embeds the project YAML and CUDA-capable inference helpers copied from `src/comment_feature_models.py`.

**You provide:** cleaned monthly Parquet on Drive (`cleaned_monthly_chunks/<subreddit>/<YYYY-MM>.parquet`).

**Notebook writes:** VM staging under `WORKDIR` then checkpoints `comment_features_ml/` back to Drive.

**On your laptop:** copy `comment_features_ml` into `data/interim/political_forums/` and run repo script:

`scripts/compute_comment_features_lexical.py --config config/political_forums_setup.yaml`

to merge lexical CPU features into final `comment_features/` for downstream metrics.

Runtime: enable **GPU** when `DEVICE = "cuda"`.


## 1) Control panel — edit embedded YAML and Drive paths


In [ ]:
# -----------------------------------------------------------------------------
# MASTER CONFIG — Optimized for Speed + Reliability
# -----------------------------------------------------------------------------
from __future__ import annotations
from pathlib import Path

WORKDIR = Path("/content/comment_ml_standalone")

DRIVE_CLEANED_MONTHLY_ROOT = "/content/drive/MyDrive/SocialAIAdoption/cleaned_monthly_chunks"
DRIVE_COMMENT_FEATURES_ML_ROOT = "/content/drive/MyDrive/SocialAIAdoption_interim/comment_features_ml_colab"

RUN_MODE = "full"
OVERWRITE = False
SUBREDDITS = ""
MONTHS = ""
DEVICE = "cuda"
BATCH_SIZE = 128
BOUNDED_MAX_TOTAL_MONTH_FILES = 1
BOUNDED_MAX_DAYS_PER_MONTH = 2
CHECKPOINT_AFTER_RUN = True
RUN_TAG = "production_run"

# Reliability and GPU stability knobs
CHECKPOINT_RETRY_ATTEMPTS = 4
CHECKPOINT_RETRY_BASE_SECONDS = 2.0
PROGRESS_LOG_FILENAME = "progress_log.jsonl"
PERPLEXITY_MAX_LENGTH = 256
ENABLE_PERPLEXITY = True

CONFIG_YAML = r'''
project:
  name: "SocialAIAdoption"
event_window:
  start_utc: "2022-11-01T00:00:00Z"
  end_utc_exclusive: "2023-05-01T00:00:00Z"
  launch_day_utc: "2022-11-30T00:00:00Z"
subreddits:
  primary:
    - cscareerquestions
    - ITCareerQuestions
    - csMajors
    - answers
    - TooAfraidToAsk
    - OutOfTheLoop
comment_features:
  detector_primary_model: "fakespot-ai/roberta-base-ai-text-detection-v1"
  detector_secondary_model: ""
  hostility_model: ""
  emotion_model: ""
  perplexity_model: ""
'''



## 2) Dependencies


In [ ]:
!pip install -q pandas pyarrow transformers torch sentencepiece pyyaml


## 3) Preflight accelerator


In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


torch 2.10.0+cu128 cuda True


In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU is ACTIVE: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT active. Please go to Runtime -> Change runtime type and select 'T4 GPU' for faster processing.")

GPU is ACTIVE: Tesla T4


## 4) Inference helpers + job runner (inlined project code — run once per session after Control)


In [ ]:
from __future__ import annotations
from typing import Any, Dict
import json
import math
import shutil
import time
import os
import gc
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
import torch

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def resolve_inference_device(device_arg: str) -> str:
    """Resolve requested device to an available backend string."""
    if device_arg == "cpu":
        return "cpu"
    if device_arg == "mps":
        return "mps"
    if device_arg == "cuda":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"


def pipeline_device_argument(device: str) -> Any:
    """Map canonical device string to transformers pipeline device argument."""
    if device == "cpu":
        return -1
    if device == "mps":
        return "mps"
    if device == "cuda":
        return 0
    return -1


def maybe_load_transformers_pipeline(task: str, model_id: str, device: str) -> Any:
    """Load transformers pipeline with conservative CUDA options or return None."""
    if not model_id:
        return None
    try:
        from transformers import pipeline

        dev = pipeline_device_argument(device)
        model_kwargs: Dict[str, Any] = {"ignore_mismatched_sizes": True}
        if device == "cuda":
            model_kwargs["torch_dtype"] = torch.float16
        return pipeline(task, model=model_id, truncation=True, device=dev, model_kwargs=model_kwargs)
    except Exception as e:
        print(f"[Warning] Failed to load model {model_id}: {e}")
        return None


def label_score_as_ai_probability(result: Dict[str, Any]) -> float:
    """Normalize pipeline label+score output to AI probability."""
    label = str(result.get("label", "")).lower()
    score = float(result.get("score", 0.0))
    if "human" in label or label.endswith("0"):
        return float(max(0.0, min(1.0, 1.0 - score)))
    return float(max(0.0, min(1.0, score)))


def batch_emotion_scores(pipe: Any, texts: list[str], batch_size: int) -> Dict[str, list[float]]:
    """Run batched emotion inference and emit aligned anger/fear/sadness/surprise scores."""
    keys = ["anger", "fear", "sadness", "surprise"]
    scores = {k: [] for k in keys}
    if pipe is None:
        for k in keys:
            scores[k] = [float("nan")] * len(texts)
        return scores

    from torch.utils.data import Dataset

    class ListDataset(Dataset):
        def __init__(self, t):
            self.t = t

        def __len__(self):
            return len(self.t)

        def __getitem__(self, i):
            return self.t[i]

    try:
        dataset = ListDataset(texts)
        for row_result in pipe(dataset, truncation=True, batch_size=batch_size, top_k=None):
            row_map = {item["label"].lower(): item["score"] for item in row_result}
            for k in keys:
                scores[k].append(row_map.get(k, 0.0))
    except Exception as e:
        print(f"[Error] Emotion batch failed: {e}")
        for k in keys:
            scores[k].extend([float("nan")] * (len(texts) - len(scores[keys[0]])))
    return scores


def init_perplexity_components(model_id: str, device: str) -> tuple[Any, Any]:
    """Load tokenizer/model for perplexity inference or return (None, None)."""
    if not model_id or not ENABLE_PERPLEXITY:
        return None, None
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
        model.eval()
        return tokenizer, model
    except Exception as e:
        print(f"[Warning] Failed to load perplexity model {model_id}: {e}")
        return None, None


def batch_perplexity(texts: list[str], tokenizer: Any, model: Any, batch_size: int, device: str) -> list[float]:
    """Compute per-text perplexity with configurable max length and safe fallbacks."""
    if tokenizer is None or model is None or not ENABLE_PERPLEXITY:
        return [float("nan")] * len(texts)
    out: list[float] = []
    for start in range(0, len(texts), max(1, int(batch_size))):
        batch = texts[start : start + max(1, int(batch_size))]
        try:
            enc = tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max(16, int(PERPLEXITY_MAX_LENGTH)),
            ).to(device)
            with torch.no_grad():
                outputs = model(**enc, labels=enc["input_ids"])
                shift_logits = outputs.logits[:, :-1, :].contiguous()
                shift_labels = enc["input_ids"][:, 1:].contiguous()
                loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
                loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                loss = loss.view(shift_labels.size(0), -1).mean(dim=1)
                out.extend(torch.exp(loss).detach().cpu().tolist())
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"[Error] Perplexity batch failed: {e}")
            out.extend([float("nan")] * len(batch))
        except Exception as e:
            print(f"[Error] Perplexity batch failed: {e}")
            out.extend([float("nan")] * len(batch))
    return out


def load_model_bundle(model_config: Dict[str, str], device: str) -> Dict[str, Any]:
    """Load all enabled models once and return reusable inference bundle."""
    bundle: Dict[str, Any] = {}
    bundle["detector_primary_pipe"] = maybe_load_transformers_pipeline("text-classification", model_config["detector_primary"], device)
    bundle["hostility_pipe"] = maybe_load_transformers_pipeline("text-classification", model_config["hostility"], device)
    bundle["emotion_pipe"] = maybe_load_transformers_pipeline("text-classification", model_config["emotion"], device)
    bundle["detector_secondary_pipe"] = maybe_load_transformers_pipeline("text-classification", model_config["detector_secondary"], device)
    bundle["ppl_tokenizer"], bundle["ppl_model"] = init_perplexity_components(model_config["perplexity"], device)
    return bundle


def release_model_bundle(bundle: Dict[str, Any]) -> None:
    """Release loaded model objects and perform bounded cleanup."""
    for key in list(bundle.keys()):
        bundle[key] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def append_ml_columns_to_records(records, texts, model_config, device, batch_size, model_bundle):
    """Compute ML columns from preloaded model bundle and mutate records in-place."""
    s1 = batch_text_classification_scores_optimized(model_bundle.get("detector_primary_pipe"), texts, batch_size)
    sh = batch_text_classification_scores_optimized(model_bundle.get("hostility_pipe"), texts, batch_size)
    se = batch_emotion_scores(model_bundle.get("emotion_pipe"), texts, batch_size)
    s2 = batch_text_classification_scores_optimized(model_bundle.get("detector_secondary_pipe"), texts, batch_size)
    sp = batch_perplexity(texts, model_bundle.get("ppl_tokenizer"), model_bundle.get("ppl_model"), batch_size, device)

    ts = datetime.now(timezone.utc).isoformat()
    for i, r in enumerate(records):
        r["detector_primary_ai_prob"] = s1[i]
        r["detector_primary_human_score"] = 1.0 - s1[i] if pd.notna(s1[i]) else float("nan")
        r["detector_secondary_ai_prob"] = s2[i]
        r["detector_secondary_human_score"] = 1.0 - s2[i] if pd.notna(s2[i]) else float("nan")
        r["hostility_score"] = sh[i]
        for k in ["anger", "fear", "sadness", "surprise"]:
            r[f"emotion_{k}"] = se[k][i]
        r["perplexity"] = sp[i]
        r["log_perplexity"] = math.log(sp[i]) if pd.notna(sp[i]) and sp[i] > 0 else float("nan")
        r["detector_primary_model_id"] = model_config["detector_primary"]
        r["detector_secondary_model_id"] = model_config["detector_secondary"]
        r["hostility_model_id"] = model_config["hostility"]
        r["emotion_model_id"] = model_config["emotion"]
        r["perplexity_model_id"] = model_config["perplexity"]
        r["device_used"] = device
        r["ml_features_computed_at_utc"] = ts


def load_config_yaml():
    """Parse embedded YAML config from control cell."""
    return yaml.safe_load(CONFIG_YAML)


def interim_chunks():
    """Return local interim root and cleaned/output chunk directories."""
    i = WORKDIR / "data" / "interim" / "political_forums"
    return i, i / "cleaned_monthly_chunks", i / "comment_features_ml"


def sync_cleaned_from_drive(src, dst, sub_a, mon_a):
    """Copy cleaned source parquets from Drive to local VM staging directory."""
    c = 0
    dst.mkdir(parents=True, exist_ok=True)
    for sd in sorted(p for p in src.iterdir() if p.is_dir()):
        if sub_a and sd.name not in sub_a:
            continue
        os_dir = dst / sd.name
        os_dir.mkdir(parents=True, exist_ok=True)
        for fp in sd.glob("*.parquet"):
            if mon_a and fp.stem not in mon_a:
                continue
            shutil.copy2(fp, os_dir / fp.name)
            c += 1
    return c


def _progress_log_path(drive_root: Path) -> Path:
    """Build progress log file path under Drive output root."""
    return drive_root / PROGRESS_LOG_FILENAME


def load_completed_outputs(drive_root: Path) -> set[str]:
    """Read progress log and return already-completed relative output paths."""
    out: set[str] = set()
    log_path = _progress_log_path(drive_root)
    if not log_path.exists():
        return out
    try:
        for line in log_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            rel = str(row.get("relative_output_path", "")).strip()
            if rel:
                out.add(rel)
    except Exception as e:
        print(f"[Warning] Could not parse progress log: {e}")
    return out


def append_progress_log(drive_root: Path, payload: Dict[str, Any]) -> None:
    """Append one JSON event line to Drive progress log file."""
    log_path = _progress_log_path(drive_root)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=True) + "\n")


def sync_single_output_to_drive(local_output_path: Path, local_root: Path, drive_root: Path, row_count: int) -> Path:
    """Atomically copy one local parquet to Drive with retries and JSONL progress logging."""
    rel = local_output_path.relative_to(local_root)
    dst = drive_root / rel
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    dst.parent.mkdir(parents=True, exist_ok=True)

    attempts = max(1, int(CHECKPOINT_RETRY_ATTEMPTS))
    base_sleep = max(0.1, float(CHECKPOINT_RETRY_BASE_SECONDS))
    for attempt in range(1, attempts + 1):
        try:
            print(f"[colab_ml_nb] checkpoint_start rel={rel} attempt={attempt}/{attempts}", flush=True)
            if tmp.exists():
                tmp.unlink()
            shutil.copy2(local_output_path, tmp)
            os.replace(tmp, dst)
            payload = {
                "event": "checkpoint_success",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "relative_output_path": str(rel),
                "rows": int(row_count),
                "attempt": int(attempt),
            }
            append_progress_log(drive_root, payload)
            print(f"[colab_ml_nb] checkpoint_success rel={rel}", flush=True)
            return dst
        except Exception as e:
            if tmp.exists():
                try:
                    tmp.unlink()
                except Exception:
                    pass
            if attempt >= attempts:
                print(f"[colab_ml_nb] checkpoint_fail rel={rel} err={e}", flush=True)
                raise
            sleep_s = base_sleep * (2 ** (attempt - 1))
            print(f"[colab_ml_nb] checkpoint_retry rel={rel} err={e} sleep_s={sleep_s:.1f}", flush=True)
            time.sleep(sleep_s)
    return dst


def sync_tree_to_drive(src, dst):
    """Reconciliation helper to copy full local output tree to Drive on demand."""
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob("*.parquet"):
        rp = dst / p.relative_to(src)
        rp.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, rp)
        n += 1
    return n


def run_ml_jobs(max_f, max_d):
    """Run monthly ML extraction with per-file Drive checkpointing and resume behavior."""
    cfg = load_config_yaml()
    _, cd, om = interim_chunks()
    subs = [s for s in cfg["subreddits"]["primary"] if not SUBREDDITS.strip() or s in SUBREDDITS.split(",")]
    f_cfg = cfg["comment_features"]
    m_cfg = {
        "detector_primary": f_cfg["detector_primary_model"],
        "detector_secondary": f_cfg["detector_secondary_model"],
        "hostility": f_cfg["hostility_model"],
        "emotion": f_cfg["emotion_model"],
        "perplexity": f_cfg["perplexity_model"],
    }

    device = resolve_inference_device(DEVICE)
    print(f"[colab_ml_nb] using device={device} batch_size={BATCH_SIZE}", flush=True)
    bundle = load_model_bundle(m_cfg, device)
    completed = load_completed_outputs(drive_output_root)

    count = 0
    try:
        for s in subs:
            for fp in sorted((cd / s).glob("*.parquet")):
                out = om / s / fp.name
                rel = str(out.relative_to(om))
                drive_out = drive_output_root / rel

                if not OVERWRITE and (drive_out.exists() or rel in completed):
                    print(f"[colab_ml_nb] skip_remote_existing rel={rel}", flush=True)
                    continue
                if not OVERWRITE and out.exists():
                    print(f"[colab_ml_nb] skip_local_existing rel={rel}", flush=True)
                    continue
                if max_f > 0 and count >= max_f:
                    return

                df = pd.read_parquet(fp).head(max_d * 500 if max_d > 0 else None)
                if max_d > 0:
                    df = df[df["date_utc"].isin(sorted(df["date_utc"].unique())[:max_d])]

                recs = df[["id", "subreddit", "date_utc"]].to_dict("records")
                append_ml_columns_to_records(
                    recs,
                    [str(b)[:2000] for b in df["body"]],
                    m_cfg,
                    device,
                    BATCH_SIZE,
                    bundle,
                )

                out.parent.mkdir(parents=True, exist_ok=True)
                out_df = pd.DataFrame(recs)
                out_df.to_parquet(out, index=False)

                if CHECKPOINT_AFTER_RUN:
                    sync_single_output_to_drive(
                        local_output_path=out,
                        local_root=om,
                        drive_root=drive_output_root,
                        row_count=len(out_df),
                    )

                count += 1
                print(f"[colab_ml_nb] progress done={count} rel={rel}", flush=True)
    finally:
        release_model_bundle(bundle)



### 4.1) Optimized Pipeline with Dataset

The following code refactors the helper to use a generator-based Dataset approach. This prevents the GPU from waiting for the CPU between batches.

In [ ]:
from torch.utils.data import Dataset


class ListDataset(Dataset):
    """Simple dataset wrapper for letting transformers pipeline batch internally."""

    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i]


def batch_text_classification_scores_optimized(pipe: Any, texts: list[str], batch_size: int) -> list[float]:
    """Run optimized batched text-classification and return normalized AI probabilities."""
    if pipe is None:
        return [float("nan")] * len(texts)

    dataset = ListDataset(texts)
    out: list[float] = []

    try:
        for results in pipe(dataset, truncation=True, batch_size=max(1, int(batch_size))):
            out.append(label_score_as_ai_probability(results))
    except Exception as e:
        print(f"[Error] Optimized pipeline failed: {e}")
        return [float("nan")] * len(texts)

    return out


print("Optimized scoring function ready.")



Optimized scoring function defined. To use it, update 'append_ml_columns_to_records' to call this version.


## 5) Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 6) Create VM paths — sync `cleaned_monthly_chunks` from Drive


In [ ]:
from pathlib import Path

_drive_cleaned = Path(DRIVE_CLEANED_MONTHLY_ROOT).expanduser().resolve()
_drive_ml_out = Path(DRIVE_COMMENT_FEATURES_ML_ROOT).expanduser().resolve()
drive_output_root = _drive_ml_out / RUN_TAG.strip() if RUN_TAG.strip() else _drive_ml_out
drive_output_root.mkdir(parents=True, exist_ok=True)

WORKDIR.mkdir(parents=True, exist_ok=True)
_, cleaned_local, features_ml_local = interim_chunks()
cleaned_local.mkdir(parents=True, exist_ok=True)
features_ml_local.mkdir(parents=True, exist_ok=True)

if not _drive_cleaned.exists():
    raise FileNotFoundError(f"Drive input not found: {_drive_cleaned}")

allow_subs = {s.strip() for s in SUBREDDITS.split(",") if s.strip()}
allow_months = {m.strip() for m in MONTHS.split(",") if m.strip()}

n_copied = sync_cleaned_from_drive(_drive_cleaned, cleaned_local, allow_subs, allow_months)
if n_copied == 0:
    raise RuntimeError("No parquet copied; check Drive path and SUBREDDITS/MONTHS filters.")


In [ ]:
import os
from pathlib import Path

def list_drive_contents(path='/content/drive/MyDrive', depth=3):
    print(f'Listing contents for: {path}')
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level < depth:
            indent = ' ' * 4 * level
            print(f'{indent}{os.path.basename(root)}/')
            sub_indent = ' ' * 4 * (level + 1)
            for d in dirs:
                print(f'{sub_indent}{d}/')
        if level >= depth:
            continue

if os.path.exists('/content/drive/MyDrive'):
    interim_path = '/content/drive/MyDrive/SocialAIAdoption_interim'
    if os.path.exists(interim_path):
        list_drive_contents(interim_path)
    else:
        list_drive_contents()
else:
    print('Drive not mounted. Please run the mount cell (sCOz3XSsusvf) first.')

Listing contents for: /content/drive/MyDrive/SocialAIAdoption_interim
SocialAIAdoption_interim/
    comment_features_ml_colab/
    comment_features_ml_colab/
        test_run/
        full_gpu_run/
        production_run/
        test_run/
            AskProgramming/
            politics/
        full_gpu_run/
            politics/
        production_run/
            PoliticalDiscussion/
            NeutralPolitics/


## 7) Run ML extraction


In [ ]:
if RUN_MODE == "bounded":
    run_ml_jobs(int(BOUNDED_MAX_TOTAL_MONTH_FILES), int(BOUNDED_MAX_DAYS_PER_MONTH))
elif RUN_MODE == "full":
    run_ml_jobs(0, 0)
else:
    raise ValueError("RUN_MODE must be bounded or full")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/848 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 8) Upload `comment_features_ml` parquet tree to Drive


In [ ]:
_, _c, ml_out = interim_chunks()

if CHECKPOINT_AFTER_RUN:
    copied = sync_tree_to_drive(ml_out, drive_output_root)
    print(f"Reconciliation checkpoint copied {copied} parquet file(s).")
else:
    print("CHECKPOINT_AFTER_RUN is False; skipping Drive reconciliation upload.")



## After Colab finishes (laptop)

1. Sync `comment_features_ml` from Drive → `data/interim/political_forums/comment_features_ml/` in your repo checkout.
2. Run: `.venv/bin/python scripts/compute_comment_features_lexical.py --config config/political_forums_setup.yaml`
3. Proceed with `prepare_event_time_metrics.py --prefer_comment_features`.


## 9) Verification


In [ ]:
import pandas as pd

_, _, ml_root = interim_chunks()

parquets = sorted(ml_root.rglob("*.parquet"))
if not parquets:
    raise FileNotFoundError(f"No output under {ml_root}")

df = pd.read_parquet(parquets[0])
need = {"id", "detector_primary_ai_prob", "device_used", "ml_features_computed_at_utc"}
missing = need - set(df.columns)
if missing:
    raise AssertionError(f"missing columns: {missing}")
print("sample:", parquets[0])
print("rows:", len(df))
print("device_used:", df["device_used"].iloc[0] if len(df) else None)
print("columns:", list(df.columns))


In [ ]:
import os
from pathlib import Path

_, _, ml_local = interim_chunks()
print(f'Checking local output directory: {ml_local}')

if ml_local.exists():
    files = list(ml_local.rglob('*.parquet'))
    if files:
        print(f'✅ Found {len(files)} result files. Your progress was saved before the disconnect.')
        for f in files[:5]:
            print(f' - {f.relative_to(WORKDIR)}')
    else:
        print('❌ Local output directory exists but is empty. You may need to re-run the extraction.')
else:
    print('❌ Local output directory does not exist.')

In [ ]:
import pandas as pd
from pathlib import Path

_, _, ml_local = interim_chunks()
check_file = ml_local / 'politics' / '2022-11.parquet'

if check_file.exists():
    df_check = pd.read_parquet(check_file)
    print(f'File: {check_file}')
    print(f'Row count: {len(df_check)}')
    print(f'Subreddit: {df_check["subreddit"].iloc[0] if len(df_check) > 0 else "N/A"}')
    if len(df_check) < 1000: # Typical full months have thousands of rows
        print('\n⚠️ This looks like a small test file from a previous bounded run.')
        print('To process the FULL month, go to the Control Panel, set OVERWRITE = True, and run the extraction cell again.')
else:
    print('File not found locally; it might only be on Drive.')

In [ ]:
import pandas as pd
import numpy as np

# Load the sample again to check for completion
_, _, ml_local = interim_chunks()
check_file = ml_local / 'politics' / '2022-11.parquet'

if check_file.exists():
    df_check = pd.read_parquet(check_file)

    # Define the columns we expect to be filled with data
    ml_cols = [
        'detector_primary_ai_prob',
        'detector_secondary_ai_prob',
        'hostility_score',
        'emotion_anger',
        'perplexity',
        'device_used'
    ]

    print("--- ML Feature Fill Status ---")
    for col in ml_cols:
        nan_count = df_check[col].isna().sum()
        fill_pct = (1 - (nan_count / len(df_check))) * 100
        print(f"{col:30} | Fill: {fill_pct:6.1f}% ({len(df_check) - nan_count}/{len(df_check)} rows)")

    print("\n--- Sample Data (First 3 rows) ---")
    display(df_check[ml_cols].head(3))

    if df_check['perplexity'].isna().all():
        print("\n❌ WARNING: Perplexity is still all NaNs. Check the GPT2 model loading status.")
    elif fill_pct > 0:
        print("\n✅ SUCCESS: Data is populating correctly!")
else:
    print('File not found to verify.')

In [ ]:
import time
import pandas as pd
from pathlib import Path

def benchmark_speed():
    _, cleaned_local, _ = interim_chunks()
    all_files = list(cleaned_local.rglob("*.parquet"))
    if not all_files:
        print("❌ No local data found. Please run cell VJzqHMCEusvg first.")
        return

    print(f"🚀 Starting benchmark on {all_files[0].name}...")
    df = pd.read_parquet(all_files[0]).head(500)
    recs = df[["id", "subreddit", "date_utc"]].to_dict("records")

    cfg = load_config_yaml()
    f_cfg = cfg["comment_features"]
    m_cfg = {
        "detector_primary": f_cfg["detector_primary_model"],
        "detector_secondary": f_cfg["detector_secondary_model"],
        "hostility": f_cfg["hostility_model"],
        "emotion": f_cfg["emotion_model"],
        "perplexity": f_cfg["perplexity_model"]
    }

    start = time.time()
    append_ml_columns_to_records(recs, [str(b)[:2000] for b in df["body"]], m_cfg, resolve_inference_device(DEVICE), BATCH_SIZE)
    end = time.time()

    duration = end - start
    rate = 500 / duration
    print(f"\n✅ Benchmark Complete!")
    print(f"Speed: {rate:.2f} comments/second")
    print(f"Time for 500 rows: {duration:.2f} seconds")

    # Estimate for a full month (approx 50k comments)
    est_full = (50000 / rate) / 3600
    print(f"Estimated time for a typical full month (50k rows): {est_full:.2f} hours")

    if est_full > 8:
        print("\n⚠️ RECOMMENDATION: Processing a full month at once may exceed Colab limits.")
        print("Change RUN_MODE = 'bounded' in the Control Panel to process in smaller daily chunks.")

try:
    benchmark_speed()
except Exception as e:
    print(f"❌ Benchmark failed: {e}\nEnsure Initialization cells (2, 3, 4) have been run.")

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# Ensure drive is mounted before checking
if not os.path.exists('/content/drive'):
    print("Drive not detected. Attempting to mount...")
    drive.mount('/content/drive')

# 1. Check Drive Output
_drive_ml_out = Path(DRIVE_COMMENT_FEATURES_ML_ROOT).expanduser().resolve()
drive_prod_path = _drive_ml_out / RUN_TAG.strip()

print(f"--- Checking Drive Storage: {drive_prod_path} ---")
if drive_prod_path.exists():
    drive_files = list(drive_prod_path.rglob('*.parquet'))
    if drive_files:
        print(f"✅ Found {len(drive_files)} files on Drive.")
        for f in drive_files[:10]:
            print(f" - {f.relative_to(drive_prod_path)}")
    else:
        print("⚠️ Drive folder exists but is empty.")
else:
    print("❌ Production folder not found on Drive.")

# 2. Check Local VM Storage
try:
    _, _, ml_local = interim_chunks()
    print(f"\n--- Checking Local VM: {ml_local} ---")
    if ml_local.exists():
        local_files = list(ml_local.rglob('*.parquet'))
        print(f"✅ Found {len(local_files)} files locally.")
    else:
        print("❌ Local output directory is empty.")
except NameError:
    print("\n❌ Kernel variables lost. Please re-run the initialization cells (Control Panel and Inference Helpers).")

In [ ]:
from pathlib import Path

try:
    checkpoint_enabled = CHECKPOINT_AFTER_RUN
except NameError:
    checkpoint_enabled = True

try:
    target_folder = drive_output_root
except NameError:
    _drive_ml_out = Path('/content/drive/MyDrive/SocialAIAdoption_interim/comment_features_ml_colab').expanduser().resolve()
    target_folder = _drive_ml_out / "production_run"

progress_log = target_folder / "progress_log.jsonl"

print(f"Incremental Checkpointing: {'ENABLED' if checkpoint_enabled else 'DISABLED'}")
print(f"Target Drive Folder: {target_folder}")
print(f"Progress Log Path: {progress_log}")
print(f"Progress Log Exists: {progress_log.exists()}")

if not checkpoint_enabled:
    print("WARNING: CHECKPOINT_AFTER_RUN is False. Set it to True to persist each file.")



### 💡 Tip: Keep Colab from Disconnecting

To prevent the session from timing out due to inactivity, you can open your browser's console (**F12** or **Ctrl+Shift+I** -> **Console** tab) and paste the following code. It will click the 'Connect' button every 60 seconds to keep the session alive:

```javascript
function ClickConnect(){
  console.log("Clicking Connect Button");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```